In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import os
import pickle
import numpy as np
import pylupnt as pnt

In [ ]:
a_max = 18000
a_min = pnt.R_MOON + 100  # 1838 km

T_max = 2 * np.pi * np.sqrt((a_max**3) / pnt.GM_MOON)
T_min = 2 * np.pi * np.sqrt((a_min**3) / pnt.GM_MOON)

a_ranges = np.linspace(a_min, a_max, 100)
T_ranges = 2 * np.pi * np.sqrt((a_ranges**3) / pnt.GM_MOON)

Tm = np.array([1 / 1, 2 / 3, 1 / 2, 2 / 5, 1 / 3, 2 / 7, 1 / 4])
Tm_str = ["2", "4/3", "1", "4/5", "2/3", "4/7", "1/2"]

a_48hr = ((48 * 3600 / (2 * np.pi)) ** 2 * pnt.GM_MOON) ** (1 / 3)
a_ref = a_48hr
print(f"SMA for 48 hr period: {a_48hr:.1f} km")
T_ref = 2 * np.pi * np.sqrt((a_ref**3) / pnt.GM_MOON)
T_vals = Tm * T_ref

plt.figure(figsize=(8, 3))
plt.plot(a_ranges, T_ranges / 3600)
plt.xlim([a_min, a_max])
plt.ylim([T_min / 3600, T_max / 3600])
plt.xlabel("Semi-major axis [km]")
plt.ylabel("Orbital period [hr]")
for i in range(len(Tm)):
    sma_T = ((T_vals[i] / (2 * np.pi)) ** 2 * pnt.GM_MOON) ** (1 / 3)
    plt.plot([sma_T, sma_T], [0, T_vals[i] / 3600], "k--")
    plt.plot([0, sma_T], [T_vals[i] / 3600, T_vals[i] / 3600], "k--")
    plt.text(a_min + 5000 - 1000 * i, T_vals[i] / 3600 + 0.5, Tm_str[i], color="r")
    plt.xlim([0, a_max])
    plt.ylim([0, T_max / 3600])
plt.grid()
plt.title("Orbital period vs Semi-major axis for lunar orbit")
plt.tight_layout()
plt.savefig(os.path.join("figs", "orbital_periods.pdf"), dpi=300)
plt.show()

In [ ]:
base_dir = "/Users/keidaiiiyama/Dropbox/Research/ResearchPapers/2025/25_09_IONGNSS/ConstellationDesign/Data"
phases = [1, 2, 3]
gen = 100
popsize = 100
lifemodel = "short"  # 'short' or 'long'
phases_str = "_".join(["{:01d}".format(p) for p in phases])
config_dir = "phase_{}_gen_{}_pop_{}_fmodel_{}".format(
    phases_str, gen, popsize, lifemodel
)

os.makedirs(os.path.join("figs", "opt", config_dir), exist_ok=True)

genmax = None
# genmax = 17  # set to None to process all generations

# list all files in config_dir
data_dir = os.path.join(base_dir, config_dir)
files = os.listdir(data_dir)
files = [f for f in files if f.startswith("gen") and f.endswith(".npz")]
files.sort()
print(files)
gennum = len(files)

Fs = []
F_alls = []
Xs = []
hvs = []

if genmax is None:
    genmax = gennum

for geni in range(genmax):
    datafile = os.path.join(data_dir, "gen_{}.npz".format(geni + 1))
    data = np.load(datafile, allow_pickle=True)

    F = data["F"]
    F_all = data["F_all"]
    X = data["X"]

    Fs.append(F)
    F_alls.append(F_all)
    Xs.append(X)
    hvs.append(data["hv"])

    # print("Gen: {}  HV: {}".format(geni, hv))

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(np.arange(min(gennum, 50)) + 1, hvs[:50], "-o")
plt.xlabel("Generation", fontsize=14, fontweight="bold")
plt.ylabel("Hypervolume", fontsize=14, fontweight="bold")
plt.title("Hypervolume over Generations")
plt.grid()
plt.xlim([0, 51])
plt.savefig(os.path.join(data_dir, "hypervolume_over_generations.pdf"), dpi=300)
plt.tight_layout()

In [ ]:
from src.constellation_design import setup_problem_config, ConstellationOptimization

if os.path.exists(os.path.join(data_dir, "config.pkl")):
    print("Loading existing problem configuration...")
    probconfig = pickle.load(open(os.path.join(data_dir, "config.pkl"), "rb"))
    sat_range_phases = probconfig["sat_range_phases"]
    opt = ConstellationOptimization(probconfig, verbose=0)

print(sat_range_phases)

## Case 1: Long lifetime 

In [ ]:
from src.postprocess import plot_history_hv

# concatenate all generations
F = np.vstack(Fs)

Xstack = []
for i in range(len(Xs)):
    Xstack.extend(Xs[i])
Xstack = np.array(Xstack)

print(F.shape)
print(Xstack.shape)

print("Total solutions before filtering: ", F.shape[0])

# get solutions with F <= 1
valid_idx = (
    (F[:, 0] <= 1.0)
    & (F[:, 1] <= 1.0)
    & (F[:, 2] <= 1.0)
    & (F[:, 3] <= 1.0)
    & (F[:, 4] <= 1.0)
    & (F[:, 5] <= 1.0)
)
print(valid_idx.shape)
F = F[valid_idx > 0, :]
X = Xstack[valid_idx > 0]

print("Total solutions: ", F.shape[0])

In [ ]:
# find solution with best the pareto fronts at stage 1
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
import matplotlib.pyplot as plt

# 1st stage pareto front
objidxs = [[0, 1], [2, 3], [4, 5]]  # indices of objectives for each stage
colors = ["tab:blue", "tab:orange", "tab:green"]


def plot_pareto_front_per_stage(F, figname=None):

    fig, ax = plt.subplots(3, 3, figsize=(15, 8))

    for stage in range(3):  # parato front stage
        nds = NonDominatedSorting()
        fronts = nds.do(F[:, objidxs[stage]], only_non_dominated_front=True)
        print("Pareto front indices for stage {}: {}".format(stage + 1, fronts))

        for stage_plot in range(3):
            sat_min = sat_range_phases[stage_plot][0]
            sat_max = sat_range_phases[stage_plot][1]
            sat_range = sat_max - sat_min
            objidx = objidxs[stage_plot]
            ax[stage, stage_plot].scatter(
                F[:, objidx[1]] * sat_range + sat_min,
                F[:, objidx[0]],
                s=10,
                color="lightgray",
            )
            ax[stage, stage_plot].scatter(
                F[fronts, objidx[1]] * sat_range + sat_min,
                F[fronts, objidx[0]],
                s=20,
                color=colors[stage],
            )
            ax[stage, stage_plot].set_xlabel(
                "Number of Satellites".format(stage_plot + 1)
            )
            ax[stage, stage_plot].set_ylabel(
                "WDOP unsatisfaction ratio".format(stage_plot + 1)
            )
            ax[stage, stage_plot].set_title(
                "Objective for stage {}".format(stage_plot + 1)
            )
            ax[stage, stage_plot].grid(True)
            ax[stage, stage_plot].set_ylim([-0.1, 1.1])

    plt.tight_layout()

    if figname is not None:
        plt.savefig(figname, dpi=300)

    plt.show()

In [ ]:
plot_pareto_front_per_stage(
    F, figname="figs/opt/{}/pareto_front_per_stage.pdf".format(config_dir)
)

In [ ]:
from pymoo.indicators.hv import Hypervolume


def plot_parato_front_among_stage(F):

    fig, ax = plt.subplots(1, 3, figsize=(15, 4))

    plt.rcParams.update({"font.size": 12})

    # dop parato fronts
    nds = NonDominatedSorting()

    # scatter dop stage 1 vs stage 2, with color representing stage 3
    fronts = nds.do(F[:, [0, 2]], only_non_dominated_front=True)
    # compute hypervolume
    hv = Hypervolume(
        ref_point=(1, 1),
        norm_ref_point=False,
        zero_to_one=True,
        ideal=(0, 0),
        nadir=(1, 1),
    )
    dop_stage1 = F[fronts, 0]
    dop_stage2 = F[fronts, 2]
    dop_stage3 = F[fronts, 4]
    sc = ax[0].scatter(dop_stage1, dop_stage2, c=dop_stage3, cmap="viridis", s=20)
    ax[0].set_xlabel("HDOP Stage 1")
    ax[0].set_ylabel("PDOP Stage 2")
    ax[0].set_title(
        "Pareto front among stages (DOP) HV={:.3f}".format(hv.do(F[:, [0, 2]]))
    )
    cbar = plt.colorbar(sc, ax=ax[0], label="PDOP Stage 3")
    # set colobar scale to 0-1
    # ax[0].set_aspect('equal', 'box')

    # scatter dop stage 1 vs stage 3, with color representing stage 2
    fronts = nds.do(F[:, [0, 4]], only_non_dominated_front=True)
    dop_stage1 = F[fronts, 0]
    dop_stage2 = F[fronts, 2]
    dop_stage3 = F[fronts, 4]
    sc = ax[1].scatter(dop_stage1, dop_stage3, c=dop_stage2, cmap="viridis", s=20)
    ax[1].set_xlabel("HDOP Stage 1")
    ax[1].set_ylabel("PDOP Stage 3")
    ax[1].set_title("Stage 1 vs Stage 3 HV={:.3f}".format(hv.do(F[:, [0, 4]])))
    plt.colorbar(sc, ax=ax[1], label="PDOP Stage 2")
    # ax[1].set_aspect('equal', 'box')

    # scatter dop stage 2 vs stage 3, with color representing stage 1
    fronts = nds.do(F[:, [2, 4]], only_non_dominated_front=True)
    dop_stage1 = F[fronts, 0]
    dop_stage2 = F[fronts, 2]
    dop_stage3 = F[fronts, 4]
    sc = ax[2].scatter(dop_stage2, dop_stage3, c=dop_stage1, cmap="viridis", s=20)
    ax[2].set_xlabel("PDOP Stage 2")
    ax[2].set_ylabel("PDOP Stage 3")
    ax[2].set_title("Stage 2 vs Stage 3 HV={:.3f}".format(hv.do(F[:, [2, 4]])))
    plt.colorbar(sc, ax=ax[2], label="HDOP Stage 1")
    # ax[2].set_aspect('equal', 'box')

    for i in range(3):
        ax[i].grid(True)
        ax[i].set_xlim([-0.05, 1.05])
        ax[i].set_ylim([-0.05, 1.05])

    plt.tight_layout()
    plt.show()


plot_parato_front_among_stage(F)

In [ ]:
from src.constellation_design import setup_hybrid_walker_constellation, print_lunanet_x
import plotly.graph_objects as go

dyn_nbody = pnt.NBodyDynamics()
dyn_nbody.set_integrator(pnt.IntegratorType.RKF45)
dyn_nbody.set_integrator_params(
    pnt.IntegratorParams(max_iter=20, abstol=1e-10, reltol=1e-10)
)
dyn_nbody.add_body(pnt.Body.Moon(2, 2))
dyn_nbody.add_body(pnt.Body.Earth())
dyn_nbody.add_body(pnt.Body.Sun())
dyn_nbody.set_time_step(60)
dyn_nbody.set_frame(pnt.MOON_CI)


def get_orbit(X, opt, config, verbose=False):

    x = opt._get_xvector(X)

    n_walker = config["n_walker"]
    n_phase = config["n_phase"]
    tspan = config["tspan"]
    et0 = config["tai0"]
    objs = config["objs"]
    dop_target = config["dop_target"]
    lent = tspan.shape[0]
    float_sma = config.get("float_sma", True)

    et = et0 + tspan

    if verbose:
        print_lunanet_x(x, config)

    x_orb_pa, x_phase, lunanet_antennas, out = setup_hybrid_walker_constellation(
        x,
        et,
        dyn_nbody,
        float_sma,
        n_walker=n_walker,
        n_phase=n_phase,
        compute_link_budget=False,
        debug=False,
    )

    x_orb_mci = out["x_orb_mci"]
    x_walker = out["walker_idxs"]

    # orbit at frozen PA frame at time 0
    x_orb_pa0 = np.zeros_like(x_orb_pa)
    for i in range(lent):
        x_orb_pa0[:, i, :] = pnt.convert_frame(
            et0, x_orb_mci[:, i, :], pnt.MOON_CI, pnt.MOON_PA
        )

    return x_orb_pa, x_orb_mci, x_orb_pa0, x_phase, x_walker


def plot_constellation(x_orb, x_phase, x_walker, phases=[0, 1, 2], figname=None):
    fig = go.Figure()

    colors = ["blue", "orange", "green"]

    idxs = []
    for phase in phases:
        idx = np.where(x_phase == phase)[0]
        idxs.extend(idx)

    x_orb_plot = x_orb[idxs]
    x_walker = x_walker[idxs]

    for walker in np.unique(x_walker):
        idxs = np.where(x_walker == walker)[0]
        pnt.plot.plot_orbits(fig, x_orb_plot[idxs], color=colors[walker])
        pnt.plot.scatter(fig, x_orb_plot[idxs, 0, :], color=colors[walker])

    pnt.plot.plot_body(
        fig,
        pnt.MOON,
        size_factor=2,
        alpha=0.5,
    )
    pnt.plot.set_view(fig, -80, 20, 2.5)
    fig.update_layout(showlegend=True, width=400, height=400)
    if figname is not None:
        fig.write_image(figname)
    fig.show()

## Stage 1 vs Stage 3 Comparison

In [ ]:
# Third stage minimum DOP solution
stage1_nsat = 5
stage3_nsat = 30
Fmax = 0.75
Fmax2 = 0.75
savefig = False

figdir = "figs/opt/{}/nsat_{}_{}/".format(config_dir, stage1_nsat, stage3_nsat)
os.makedirs(figdir, exist_ok=True)

if not os.path.exists(figdir):
    os.makedirs(figdir)

if stage3_nsat is not None:
    max_sat_phase3 = sat_range_phases[2][1]
    min_sat_phase3 = sat_range_phases[2][0]
    sat_range3 = max_sat_phase3 - min_sat_phase3

    max_sat_phase1 = sat_range_phases[0][1]
    min_sat_phase1 = sat_range_phases[0][0]
    sat_range1 = max_sat_phase1 - min_sat_phase1

    valid_idx = F[:, 5] == (stage3_nsat - min_sat_phase3) / sat_range3
    valid_idx = valid_idx & (F[:, 1] == (stage1_nsat - min_sat_phase1) / sat_range1)
    valid_idx = (
        valid_idx & (F[:, 4] <= Fmax) & (F[:, 0] <= Fmax) & (F[:, 2] <= Fmax2)
    )  # also ensure both DOP are below 0.5
    Fvalid = F[valid_idx, :]
    Xvalid = X[valid_idx]
else:
    Fvalid = F.copy()
    Xvalid = X.copy()

stage1_min_idx = np.argmin(Fvalid[:, 0])
stage3_min_idx = np.argmin(Fvalid[:, 4])

# plot F[:, 0] vs F[:, 4] with F[:, 5] as text
plt.figure(figsize=(6, 4))
plt.scatter(Fvalid[:, 0], Fvalid[:, 4], s=10, color="gray")
# add text for PDOP stage 2
# for i in range(Fvalid.shape[0]):
#     plt.text(Fvalid[i, 0], Fvalid[i, 4], "{0:.2f}".format(Fvalid[i, 2]), fontsize=8, color='black')
plt.xlabel("WDOP unsatisfaction ratio Stage 1", fontsize=14, fontweight="bold")
plt.ylabel("WDOP unsatisfaction ratio Stage 3", fontsize=14, fontweight="bold")
plt.scatter(
    Fvalid[stage1_min_idx, 0],
    Fvalid[stage1_min_idx, 4],
    s=20,
    color="red",
    label="Min Stage 1",
)
plt.scatter(
    Fvalid[stage3_min_idx, 0],
    Fvalid[stage3_min_idx, 4],
    s=20,
    color="green",
    label="Min Stage 3",
)
plt.legend()
plt.title(
    "Filtered Solutions with Stage1: {} sat  Stage3: {} sat".format(
        stage1_nsat, stage3_nsat
    ),
    fontsize=14,
    fontweight="bold",
)
plt.grid()
plt.xlim([-0.1, 1.1])
plt.ylim([-0.1, 1.1])
plt.tight_layout()
plt.savefig(
    os.path.join(
        figdir,
        "filtered_solutions_stage1_{}_stage3_{}.pdf".format(stage1_nsat, stage3_nsat),
    ),
    dpi=300,
)

### Stage 3 Optimal Solution

In [ ]:
dopmin_idx = np.argmin(Fvalid[:, 4])
x_orb_pa, x_orb_mci, x_orb_pa0, x_phase3, x_walker3 = get_orbit(
    Xvalid[dopmin_idx], opt, probconfig, verbose=True
)

lent = x_orb_pa0.shape[1]
x_orb_pa0 = x_orb_pa0[:, : int(lent / 2), :]

print("-------------------------------")
print("Sat Nums:", x_orb_pa.shape[0])
for phase in range(3):
    print(
        "Phase {}   Sat Nums: {}  WDOP: {}".format(
            phase, np.sum(x_phase3 <= phase), 1 - Fvalid[dopmin_idx, phase * 2]
        )
    )
print("-------------------------------")

if savefig:
    plot_constellation(
        x_orb_pa0, x_phase3, x_walker3, phases=[0], figname=figdir + "phase3opt_p1.pdf"
    )
    plot_constellation(
        x_orb_pa0,
        x_phase3,
        x_walker3,
        phases=[0, 1],
        figname=figdir + "phase3opt_p2.pdf",
    )
    plot_constellation(
        x_orb_pa0,
        x_phase3,
        x_walker3,
        phases=[0, 1, 2],
        figname=figdir + "phase3opt_p3.pdf",
    )
else:
    plot_constellation(x_orb_pa0, x_phase3, x_walker3, phases=[0])
    plot_constellation(x_orb_pa0, x_phase3, x_walker3, phases=[0, 1])
    plot_constellation(x_orb_pa0, x_phase3, x_walker3, phases=[0, 1, 2])

### Stage1 Optimal Solution

In [ ]:
dopmin_idx = np.argmin(Fvalid[:, 0])

x_orb_pa, x_orb_mci, x_orb_pa0, x_phase1, x_walker1 = get_orbit(
    Xvalid[dopmin_idx], opt, probconfig, verbose=True
)
print(Fvalid[dopmin_idx, :])

lent = x_orb_pa0.shape[1]
x_orb_pa0 = x_orb_pa0[:, : int(lent / 2), :]

print("-------------------------------")
print("Sat Nums:", x_orb_pa.shape[0])
for phase in range(3):
    print(
        "Phase {}   Sat Nums: {}  WDOP: {}".format(
            phase, np.sum(x_phase1 <= phase), 1 - Fvalid[dopmin_idx, phase * 2]
        )
    )
print("-------------------------------")

if savefig:
    plot_constellation(
        x_orb_pa0, x_phase1, x_walker1, phases=[0], figname=figdir + "phase1opt_p1.pdf"
    )
    plot_constellation(
        x_orb_pa0,
        x_phase1,
        x_walker1,
        phases=[0, 1],
        figname=figdir + "phase1opt_p2.pdf",
    )
    plot_constellation(
        x_orb_pa0,
        x_phase1,
        x_walker1,
        phases=[0, 1, 2],
        figname=figdir + "phase1opt_p3.pdf",
    )
else:
    plot_constellation(x_orb_pa0, x_phase1, x_walker1, phases=[0])
    plot_constellation(x_orb_pa0, x_phase1, x_walker1, phases=[0, 1])
    plot_constellation(x_orb_pa0, x_phase1, x_walker1, phases=[0, 1, 2])

In [ ]:
print(x_walker1)

print(x_phase1)

print(len(x_phase1))

In [ ]:
from src.constellation_design import int_to_sma

# print latex table
# Ex:
# Type & Walker & Semi-major axis & Eccentricity & $\Omega_0$ & Stage 1 & Stage 2 & Stage 3 \\
# \hline \hline
# \multirow{3}{*}{Stage 1 Pareto} & 1 & 11031 & 0.696 & 180.0 & 6 & 1 & 1 \\
# & 2 & 5075.57 & 0.031 & 268.0 & 0 & 5 & 8 \\
# & 3 & 11031 & 0.696 & 253.0 & 0 & 0 & 2 \\  \hline
# \multirow{3}{*}{Stage 3 Pareto}
# & 1 & 9950 & 0.547 & 273.0 & 4 & 0 & 3 \\
# & 2 & 6300 & 0.025 & 268.0 & 2 & 7 & 4 \\
# & 3 & 12300 & 0.473 & 278.0 & 0 & 2 & 1 \\ \hline
N_W = 7

stage1_opt_idx = np.argmin(Fvalid[:, 0])
stage3_opt_idx = np.argmin(Fvalid[:, 4])

filename = os.path.join(
    figdir, "opt_summary_stage1_{}_stage3_{}.txt".format(stage1_nsat, stage3_nsat)
)
with open(filename, "w") as f:

    # Table 1 -------------------------------------------------------
    f.write("\\begin{tabular}{|c|cc|ccccc|ccc|}\n")
    f.write(
        "Type & Walker & Num Plane & Num Sat/Plane & $a_k$ & $e_k$ & $w$ & $\\Omega_0$ & Stage 1 & Stage 2 & Stage 3 \\\\\n"
    )
    f.write("\\hline \\hline\n")

    # Stage 1 Pareto
    f.write("\\multirow{3}{*}{Stage 1 Pareto}")
    for i in range(3):
        f.write(
            " & {:d} & {:d} & {:d} & {:.1f} & {:.3f} & {:.0f} & {:.1f} & {:d} & {:d} & {:d} \\\\\n".format(
                i + 1,
                int(opt._get_xvector(Xvalid[stage1_opt_idx])[3 + i * N_W]),
                int(opt._get_xvector(Xvalid[stage1_opt_idx])[4 + i * N_W]),
                int_to_sma(int(opt._get_xvector(Xvalid[stage1_opt_idx])[0 + i * N_W])),
                opt._get_xvector(Xvalid[stage1_opt_idx])[1 + i * N_W],
                opt._get_xvector(Xvalid[stage1_opt_idx])[2 + i * N_W],
                opt._get_xvector(Xvalid[stage1_opt_idx])[3 + i * N_W] * 180 / np.pi,
                np.sum((x_phase1 == 0) * (x_walker1 == i)),
                np.sum((x_phase1 == 1) * (x_walker1 == i)),
                np.sum((x_phase1 == 2) * (x_walker1 == i)),
            )
        )

    # stage 3 Pareto
    f.write("\\hline\n")
    f.write("\\multirow{3}{*}{Stage 3 Pareto}")
    for i in range(3):
        f.write(
            " & {:d} & {:d} & {:d} & {:.1f} & {:.3f} & {:.0f} & {:.1f} & {:d} & {:d} & {:d} \\\\\n".format(
                i + 1,
                int(opt._get_xvector(Xvalid[stage3_opt_idx])[3 + i * N_W]),
                int(opt._get_xvector(Xvalid[stage3_opt_idx])[4 + i * N_W]),
                int_to_sma(int(opt._get_xvector(Xvalid[stage3_opt_idx])[0 + i * N_W])),
                opt._get_xvector(Xvalid[stage3_opt_idx])[1 + i * N_W],
                opt._get_xvector(Xvalid[stage3_opt_idx])[2 + i * N_W],
                opt._get_xvector(Xvalid[stage3_opt_idx])[3 + i * N_W] * 180 / np.pi,
                np.sum((x_phase3 == 0) * (x_walker3 == i)),
                np.sum((x_phase3 == 1) * (x_walker3 == i)),
                np.sum((x_phase3 == 2) * (x_walker3 == i)),
            )
        )
    f.write("\\hline\n")
    f.write("\\end{tabular}\n")

    # empty lines
    f.write("\n\n\n")

    # Table 2 -------------------------------------------------------
    # Number of satellites and DOP values for each stage
    f.write("\\begin{tabular}{|c|c c c|c c c|}\n")
    f.write(
        " & \\multicolumn{3}{c|}{Number of Satellites} & \\multicolumn{3}{c|}{DOP under Target} \\\\\n"
    )
    f.write("Type & Stage 1 & Stage 2 & Stage 3 & Stage 1 & Stage 2 & Stage 3 \\\\\n")
    f.write("\\hline \\hline\n")
    f.write(
        "Stage 1 Pareto & {:d} & {:d} & {:d} & {:.3f} & {:.3f} & {:.3f} \\\\\n".format(
            np.sum(x_phase1 == 0),
            np.sum(x_phase1 == 1),
            np.sum(x_phase1 == 2),
            1 - Fvalid[stage1_opt_idx, 0],
            1 - Fvalid[stage1_opt_idx, 2],
            1 - Fvalid[stage1_opt_idx, 4],
        )
    )
    f.write(
        "Stage 3 Pareto & {:d} & {:d} & {:d} & {:.3f} & {:.3f} & {:.3f} \\\\\n".format(
            np.sum(x_phase3 == 0),
            np.sum(x_phase3 == 1),
            np.sum(x_phase3 == 2),
            1 - Fvalid[stage3_opt_idx, 0],
            1 - Fvalid[stage3_opt_idx, 2],
            1 - Fvalid[stage3_opt_idx, 4],
        )
    )
    f.write("\\hline\n")
    f.write("\\end{tabular}\n")
    f.write("\n")

# show textfile
with open(filename, "r") as f:
    print(f.read())

### Others

In [ ]:
T_ant = 100.0  # [K] Antenna temperature
cable_loss_before_LNA = 1.0  # [dB]
LNA_NF = 3.0  # [dB]
LNA_gain = 30.0  # [dB]
cable_loss_after_LNA = 10.0  # [dB]
T0 = 290.0  # [K]

# Convert to linear units
LNA_F = 10 ** (LNA_NF / 10)
L1 = 10 ** (cable_loss_before_LNA / 10)
L2 = 10 ** (cable_loss_after_LNA / 10)
G_LNA = 10 ** (LNA_gain / 10)

# Effective noise temperature
T_LNA = T0 * (LNA_F - 1)
T_cable1 = T0 * (L1 - 1)
T_anteff = L1 * T_ant
T_cable2 = T0 * (L2 - 1)
Teff = T_anteff + T_cable1 + T_LNA / L1 + T_cable2 / (L1 * G_LNA)  # [K]

print("T_eff = {:.2f} K".format(Teff))

lambda_c = 58.61e-3  # [km] code wavelength
C = 299792.458  # [km/s] speed of light
sRc = C / lambda_c  # chip rate   C/Rc = lambda_c
print("chip rate = {:.2f} kcps".format(sRc))
print("chip period = {:.2e} s".format(1 / sRc))